## Open government data provided by **opendata.swiss**

_Automatically generated Python starter code for dataset_ **1274344e-8cbb-4d1c-a31b-588e741ccc9d**


## Dataset

# Plan directeur cantonal - Centralités à développer &#40;points&#41;


## Description

Pierre angulaire de l&#x27;aménagement du territoire, le plan directeur cantonal pose les grands principes et les conditions de mise en oeuvre de la politique d&#x27;aménagement du canton de Genève. Soumis à l&#x27;approbation de la Confédération, le plan directeur cantonal 2030 &#40;PDCn&#41; assure la coordination et la cohérence entre les politiques sectorielles, il détermine les mesures d&#x27;aménagement nécessaires en matière d&#x27;urbanisation, de mobilité, de gestion de l&#x27;espace rural, des milieux naturels et des ressources. Engageant pour les autorités pour lesquelles il a force obligatoire, il règle la coordination entre les politiques d&#x27;aménagement du canton et de la Confédération, des cantons voisins et des régions limitrophes. Il fournit le cadre à l&#x27;aménagement local et aux activités qui sont de la compétence des communes. L&#x27;enjeu majeur du PDCn est de proposer un projet de territoire qui respecte l&#x27;équilibre entre le développement des activités humaines et une gestion durable du territoire. Il s&#x27;agit d&#x27;offrir une réponse aux besoins en logements, activités, mobilité, équipements, services, loisirs et espaces publics, tout en préservant et valorisant patrimoine bâti, paysager et naturel, les terres cultivables et la qualité de vie. Le PDCn a été adopté par le Grand Conseil le 20 septembre 2013 et approuvé par le Conseil fédéral le 29 avril 2015. Il a fait l&#x27;objet d&#x27;une première mise à jour adoptée par le Grand Conseil le 10 avril 2019 et approuvée par la Confédération le 18 janvier 2021. Cette première mise à jour a permis d&#x27;adapter le PDCn aux nouvelles directives fédérales accompagnant l&#x27;entrée en vigueur de la loi sur l&#x27;aménagement du territoire &#40;LAT1&#41; et de lever certaines réserves émises par la Confédération lors de son approbation en 2015.


## Dataset links

[View this dataset in the data catalogue](https://opendata.swiss/de/dataset/plan-directeur-cantonal-centralites-a-developper-points)


## Metadata

- **identifier:** PDCN&#95;CENTRALITES&#95;P@canton-geneve
- **publisher.name:** Office de l&#x27;urbanisme
- **organization.display&#95;name:** Kanton Genf
- **organization.url:** 
- **maintainer:** 
- **maintainer&#95;email:** 
- **keywords:** 
- **issued:** 2025-06-12 00:00:00
- **metadata&#95;created:** 2026-01-12T16:06:28.929634
- **metadata&#95;modified:** 2026-10-31T03:54:58.615121
- **Resource id:** 72b9836e-8ad2-4277-a5eb-15afac21ea41
- **Resource name:** CSV
- **Resource package&#95;id:** 1274344e-8cbb-4d1c-a31b-588e741ccc9d
- **Resource description:** Service téléchargement extracteur
- **Resource issued:** 2025-06-12 00:00:00
- **Resource modified:** 2026-10-29 00:00:00
- **Resource license:** https://opendata.swiss/terms-of-use&#35;terms&#95;open
- **Resource rights:** https://opendata.swiss/terms-of-use&#35;terms&#95;open


## Imports and helper functions

This template targets pandas (3.x), seaborn (0.13+), matplotlib (3.x), and requests (2.x); packages are not installed or pinned by the notebook. Check the versions below when sharing results.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
print(f"pandas {pd.__version__}; seaborn {sns.__version__}")

In [ ]:
from io import BufferedReader
from typing import BinaryIO

import requests

USER_AGENT = "OpendataSwiss-StarterCode-Python/1.0"
HTTP_TIMEOUT = 30


def get_dataset(
    url: str | BinaryIO,
    *,
    sep: str | None = None,
    drop_empty_unnamed: bool = True,
    **kwargs: object,
) -> pd.DataFrame:
    """Read one CSV resource; preserve data and report parsing failures."""
    if isinstance(url, str) and url.startswith(("https://", "http://")):
        with requests.get(
            url, headers={"User-Agent": USER_AGENT}, timeout=HTTP_TIMEOUT, stream=True
        ) as response:
            response.raise_for_status()
            # Decode HTTP compression while streaming CSV bytes into pandas.
            response.raw.decode_content = True
            # Let the buffer finish reading after the HTTP connection reaches EOF.
            response.raw.auto_close = False
            with BufferedReader(response.raw) as source:
                # Detect file compression after HTTP compression has been decoded.
                if kwargs.get("compression", "infer") == "infer" and source.peek(2).startswith(
                    b"\x1f\x8b"
                ):
                    kwargs["compression"] = "gzip"
                return get_dataset(source, sep=sep, drop_empty_unnamed=drop_empty_unnamed, **kwargs)
    data = pd.read_csv(url, sep=sep, engine="python", dtype_backend="numpy_nullable", **kwargs)
    # Extra fields can make pandas silently consume leading values as an index.
    if kwargs.get("index_col") is None and not isinstance(data.index, pd.RangeIndex):
        raise pd.errors.ParserError(
            "CSV rows contain unexpected index fields. Check the delimiter and header; "
            "set index_col explicitly only if these fields are an intentional index."
        )
    if drop_empty_unnamed and len(data) > 0:
        # Blank CSV headers (often from trailing separators) become "Unnamed: n".
        unnamed = data.columns.astype(str).str.fullmatch(r"Unnamed: \d+(?:\.\d+)?")
        empty_unnamed = unnamed & data.isna().all()
        if empty_unnamed.any():
            print("Removed empty unnamed columns:", data.columns[empty_unnamed].tolist())
            data = data.loc[:, ~empty_unnamed]
    return data

## Load data

Selected resource: **CSV** (CSV). This notebook loads only this resource.

For CSV, the separator is guessed. If parsing looks wrong (especially for a single-column file), set `sep=","` or `sep=";"` in `get_dataset(...)`. Check inferred types, missing-value codes, decimal marks, and identifiers with leading zeros; use `dtype`, `na_values`, `decimal`, or `encoding` as needed.

Unexpected index inference raises a parsing error to prevent values shifting under the wrong headers. Check the delimiter and header, or set `index_col` explicitly for an intentional index.

Fully empty columns named `Unnamed: n` are removed on import. Named empty columns and unnamed columns containing values are kept. Use `drop_empty_unnamed=False` to retain every column, including intentionally named `Unnamed: n` fields.


In [ ]:
df = get_dataset("https://extract.sitg.ge.ch/files/PDCN_CENTRALITES_P-csv.zip")

## Analyze data

Inspect before cleaning: these cells leave `df` unchanged. Duplicate rows and missing values may be meaningful; check the metadata before removing or filling them.


In [ ]:
print(f"{len(df):,} rows × {len(df.columns):,} columns")
print(f"{df.duplicated().sum():,} duplicate rows after the first occurrence")
df.info()
display(df.head())

In [ ]:
# Column types, completeness, and distinct non-missing values.
profile = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=True),
    }
).sort_values("missing_pct", ascending=False)
display(profile.round(1))

In [ ]:
# Show up to 20 columns with missing values; percentages use all rows.
missing = profile.loc[profile["missing"] > 0].head(20).rename_axis("column").reset_index()
if not missing.empty:
    fig, ax = plt.subplots(figsize=(8, max(2, len(missing) * 0.3)))
    sns.barplot(data=missing, x="missing_pct", y="column", color="steelblue", errorbar=None, ax=ax)
    ax.set(xlim=(0, 100), xlabel="Missing values (%)", ylabel="", title="Missingness by column")
    fig.tight_layout()
    plt.show()
else:
    print("No missing values to plot (or the dataset has no rows).")

In [ ]:
# Summarize numeric and non-numeric columns, including pandas string types.
if not df.empty:
    display(df.describe(include="all").T)
else:
    print("No data to summarize.")

In [ ]:
# Plot up to four numeric columns. Exclude missing/infinite values only from plots.
numeric = df.select_dtypes(include="number")
for column in numeric.columns[:4]:
    values = numeric[column].dropna()
    values = values.loc[~values.isin([float("inf"), -float("inf")])]
    if not values.empty:
        fig, ax = plt.subplots(figsize=(7, 3))
        sns.histplot(x=values, bins=30, ax=ax)
        ax.set(title=column, xlabel=column, ylabel="Count")
        fig.tight_layout()
        plt.show()

In [ ]:
# Top 10 values for up to three categorical columns, including missing values.
categorical = df.select_dtypes(include=["string", "object", "category", "bool"])
for column in categorical.columns[:3]:
    print(column)
    display(df[column].value_counts(dropna=False).head(10).to_frame("count"))

### Next steps

Choose columns from `df.columns`: compare groups, inspect a time trend, or plot two numeric variables. Adapt the commented examples below; use the metadata to decide which comparisons make sense.


In [ ]:
# Group summary (replace column names):
# df.groupby("group", dropna=False)["value"].agg(["count", "mean", "median"])

# Parse a date explicitly, then inspect a time trend:
# dated = df.assign(date=pd.to_datetime(df["date"], format="%Y-%m-%d", errors="raise"))
# dated.groupby("date")["value"].mean().plot(ylabel="Mean value")

# Relationship between two numeric variables:
# sns.scatterplot(data=df, x="x_column", y="y_column", alpha=0.5)

**Questions about the data?** pierre.stoenesco@etat.ge.ch &#124; Office de l&#x27;urbanisme
